# ✈️ Predicción de retrasos en vuelos comerciales de EE.UU. (jun-2025 a may-2026)
### Proyecto final · Machine Learning · Universidad EIA
**Rama:** `CARLOS-BEDOYA` · **Autor de esta versión:** Carlos Bedoya

**Pregunta de negocio:** con la información que existe **antes de que el avión salga** (itinerario, ruta, aerolínea, fecha), ¿podemos anticipar si un vuelo va a **llegar con 15 minutos o más de retraso**?

Este notebook recorre, en orden, **los 6 pasos de la sesión 6** que pide la rúbrica:

| Paso | Qué se muestra | Sección |
|---|---|---|
| 1 · ¿ML? | El problema y una **regla a mano** como punto de comparación | §1 |
| 2 · Datos | De qué página salen, cómo se descargan, qué es una fila y cuál es la etiqueta | §2 |
| 3 · Limpieza | Faltantes, duplicados, cancelados y **fugas de información** | §3 |
| 4 · Columnas | Cuáles entran, cuáles se crean y cuáles se descartan (tabla antes/después) | §4 |
| 5 · Modelos | Línea base, regla manual y 5 familias de modelos con **la misma validación** | §5 |
| 6 · Evaluación | Tabla final, métricas en prueba y **elección argumentada** | §6 |

> **Cómo correrlo:** `Kernel → Restart & Run All`. Semilla fija (42): los números coinciden en cada corrida.
> Si todavía no tienes la muestra, la celda del Paso 2 la crea a partir de los 12 CSV de `data/raw/`.

## 0 · Preparación del entorno

In [ ]:
# --- Librerías estándar de ciencia de datos -----------------------------------
import warnings                      # para silenciar avisos que no afectan los resultados
import numpy as np                   # cálculo numérico
import pandas as pd                  # tablas (DataFrames)
import matplotlib.pyplot as plt      # gráficas
import seaborn as sns                # gráficas estadísticas con mejor estilo

# --- scikit-learn: lo que se usa directamente en este notebook ----------------
from sklearn.base import clone                       # copia "en blanco" de un modelo
from sklearn.linear_model import LogisticRegression  # para la demostración de fuga
from sklearn.metrics import roc_auc_score            # métrica principal
from sklearn.pipeline import make_pipeline           # pipeline corto para la demostración
from sklearn.preprocessing import StandardScaler     # escalado para la demostración
from sklearn.impute import SimpleImputer             # imputación para la demostración

# --- Nuestro propio código (carpeta src/) --------------------------------------
from src import config                                           # rutas, semilla y listas de columnas
from src.data.download_data import construir_muestra             # Paso 2: descarga + muestra
from src.data.load_data import cargar_muestra                    # Paso 2: leer la muestra
from src.features.build_features import (limpiar_filas,          # Paso 3: limpieza de filas
                                         convertir_fecha,        # Paso 3: fecha de texto a fecha real
                                         crear_columnas,         # Paso 4: columnas nuevas
                                         descartar_fugas)        # Paso 3: quitar fugas
from src.features.imputation import comparar_imputadores         # Paso 3: elegir imputador
from src.models.pipeline import crear_pipeline                   # Pasos 4-5: Pipeline completo
from src.models.experiment import (separar_temporal, regla_manual, definir_modelos,
                                   validar_modelo, elegir_campeon, ajustar_hiperparametros)
from src.models.evaluate import (metricas, graficar_comparacion_cv, graficar_curvas,
                                 graficar_matrices, importancia_columnas)

warnings.filterwarnings("ignore")                      # avisos de versiones, no de errores
pd.set_option("display.max_columns", 50)               # mostrar todas las columnas
pd.set_option("display.float_format", "{:,.4f}".format)  # 4 decimales en las tablas
sns.set_theme(style="whitegrid")                       # estilo de las gráficas
np.random.seed(config.SEMILLA)                         # semilla global (42) para reproducibilidad
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)  # carpeta donde se guardan las figuras
print("Semilla:", config.SEMILLA, "| Carpeta del proyecto:", config.BASE_DIR)

---
## 1 · ¿Por qué Machine Learning? (y la regla a mano)

**El problema.** EE.UU. opera más de 7 millones de vuelos domésticos al año y cerca de 1 de cada 5 llega 15 minutos tarde o más. Un retraso cuesta combustible, horas extra de tripulación, conexiones perdidas y compensaciones a pasajeros. Saber **con anticipación** qué vuelos tienen más riesgo permite reforzar tripulaciones, reprogramar conexiones o avisar a los pasajeros.

**¿Por qué no basta una regla?** El retraso depende de muchos factores que se combinan de forma no lineal:
- **Hora del día:** un avión hace varios trayectos; un retraso de la mañana se propaga a la tarde (*efecto cascada*).
- **Temporada:** tormentas en verano, nieve en invierno, picos de viajeros en Navidad y *Spring Break*.
- **Aeropuerto y aerolínea:** los *hubs* (Atlanta, Chicago O'Hare, Dallas, Denver...) concentran congestión.

**Regla a mano (sin ML)** — lo que diría alguien con sentido común:

> *“El vuelo se retrasa si sale a las 4:00 p. m. o después **y** su origen o destino es uno de los 10 hubs más congestionados.”*

Esta regla es nuestro **punto de comparación**: el modelo de ML solo vale la pena si la supera.
Junto a ella está la **línea base**: decir siempre “puntual”, que ya acierta cerca del 78 % (¡pero no detecta ningún retraso!).

In [ ]:
# La regla manual está programada en src/models/experiment.py. Así se ve por dentro:
import inspect                                   # permite imprimir el código fuente de una función
print(inspect.getsource(regla_manual))
print("Hubs usados:", sorted(config.HUBS))

---
## 2 · Datos: de dónde salen y cómo se descargan

| | |
|---|---|
| **Entidad que publica** | *Bureau of Transportation Statistics (BTS)*, Departamento de Transporte de EE.UU. |
| **Tabla oficial** | *Reporting Carrier On-Time Performance* — [transtats.bts.gov](https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ) |
| **Dónde la descargamos** | Kaggle: [2021-2026 US Flights](https://www.kaggle.com/datasets/umerhafeezsiddiqui/2021-2026-us-flights) (un CSV por mes) |
| **Licencia** | Obra del Gobierno Federal de EE.UU. → dominio público |
| **Periodo de estudio** | 1 jun 2025 – 31 may 2026: **12 meses = un ciclo estacional completo** |
| **Cómo se descarga** | API de Kaggle (`src/data/download_data.py`) o a mano desde la página |
| **Qué es una fila** | **Un vuelo comercial doméstico programado** (aerolínea + número + fecha + ruta) |

**Complejidad de los datos.** No es un CSV limpio: son **12 archivos crudos** (~7,8 millones de filas y ~80 MB cada uno) que se auditan uno por uno, se filtran (cancelados, desviados, sin etiqueta) y se unen en **una tabla principal armada por nosotros**. Para que el proyecto corra en vivo en un portátil se toma **el 1 % de los vuelos operados de cada mes** al azar con semilla 42: cada mes aporta en proporción a su tráfico, así que la muestra conserva la estacionalidad.

In [ ]:
# Si la muestra no existe todavía, se construye desde los 12 CSV de data/raw/
# (y si tampoco están los CSV, se intentan descargar de Kaggle con tu token).
if not config.SAMPLE_PARQUET_PATH.exists():
    construir_muestra(fraccion=config.FRACCION_MUESTRA)

muestra = cargar_muestra()                         # tabla principal: 1 fila = 1 vuelo operado
print(f"Muestra: {muestra.shape[0]:,} filas x {muestra.shape[1]} columnas")
muestra.head()

### 2.1 Auditoría de los 12 archivos crudos
Esta tabla se calcula sobre **todos** los vuelos de cada mes (no solo la muestra) al leer los CSV. Es la evidencia de lo que hubo que limpiar.

In [ ]:
auditoria = pd.read_csv(config.REPORTS_DIR / "auditoria_mensual_cruda.csv")   # una fila por mes
display(auditoria)

# Totales del año: cuántos vuelos crudos hay y cuántos se pierden por cada motivo
totales = auditoria[["filas", "cancelados", "desviados", "nulos_target",
                     "operados_con_target", "duplicados_clave_vuelo", "nulos_crs_elapsed"]].sum()
print("\nTOTALES DEL AÑO")
print(totales.apply(lambda v: f"{v:,.0f}").to_string())

In [ ]:
# ¿La muestra respeta el peso de cada mes y su tasa de retraso?
cobertura = pd.read_csv(config.REPORTS_DIR / "muestra_vs_mes.csv")
display(cobertura)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
mes_corto = cobertura["archivo"].str.replace(".csv", "", regex=False)          # "Jun-2025"
ax[0].bar(mes_corto, cobertura["muestra"], color="#4C72B0")                   # filas por mes en la muestra
ax[0].set_title("Vuelos de cada mes en la muestra (1 %)")
ax[0].tick_params(axis="x", rotation=45)
ax[1].plot(mes_corto, cobertura["tasa_retraso_operados"], "o-", label="Todos los vuelos del mes")
ax[1].plot(mes_corto, cobertura["tasa_retraso_muestra"], "s--", label="Muestra")
ax[1].set_title("Tasa de retraso: población vs muestra")
ax[1].tick_params(axis="x", rotation=45)
ax[1].legend()
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "00_cobertura_mensual.png", dpi=120)
plt.show()

### 2.2 La etiqueta (variable objetivo)

$$ y = \begin{cases} 1 & \text{si } ARR\_DELAY \ge 15 \text{ min (retraso significativo)} \\ 0 & \text{si } ARR\_DELAY < 15 \text{ min (a tiempo o adelantado)} \end{cases} $$

Los 15 minutos son el **estándar oficial de la FAA y el DOT**: por debajo, el vuelo se considera *on-time*. La columna `ARR_DEL15` ya trae esta definición. Es un problema de **clasificación binaria supervisada** y **desbalanceada** (pocos “1”).

In [ ]:
conteo = muestra[config.TARGET].value_counts().sort_index()         # cuántos 0 y cuántos 1
print(pd.DataFrame({"vuelos": conteo, "porcentaje": conteo / conteo.sum() * 100})
        .rename(index={0.0: "0 = a tiempo", 1.0: "1 = retraso"}))

---
## 3 · Limpieza: faltantes, duplicados y fugas

### 3.1 Limpieza de filas
`limpiar_filas()` (en `src/features/build_features.py`): quita cancelados/desviados y filas sin etiqueta (no se puede imputar *si llegó tarde*), elimina duplicados de la clave del vuelo, convierte la fecha de texto a fecha real y **ordena cronológicamente**.

**Formatos:** `FL_DATE` llega con **dos formatos mezclados** según el mes (`6/30/2025 12:00:00 AM` y `06-01-25 00:00`); `convertir_fecha()` prueba cada formato exacto y unifica todo. `CRS_DEP_TIME` viene en HHMM y el BTS escribe la medianoche como `2400` (se lee como 0 h).

In [ ]:
limpia = limpiar_filas(muestra)            # aplica los 5 pasos de limpieza descritos arriba
print(f"Filas antes: {len(muestra):,} | filas después: {len(limpia):,}")
print(f"Rango de fechas: {limpia['FL_DATE'].min():%Y-%m-%d} a {limpia['FL_DATE'].max():%Y-%m-%d}")

### 3.2 Valores faltantes en la tabla limpia

In [ ]:
faltantes = limpia.isna().sum()                                  # vacíos por columna
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
tabla_faltantes = pd.DataFrame({"vacios": faltantes,
                                "porcentaje": faltantes / len(limpia) * 100,
                                "es_fuga": [c in config.COLUMNAS_FUGA for c in faltantes.index]})
tabla_faltantes

**Lectura:** casi todos los vacíos están en columnas de **causas de retraso** (`CARRIER_DELAY`, `WEATHER_DELAY`...): el BTS solo las llena cuando hubo retraso. Igual se descartan por ser **fugas**. En las columnas del itinerario casi no faltan datos; aun así el modelo debe saber qué hacer si llega un vacío en producción → se elige el imputador **comparando errores** (siguiente celda).

### 3.3 ¿Con qué se rellena la duración programada? Se compara, no se supone
Igual que con la edad en el Titanic: se **esconde el 20 %** de los valores conocidos de `CRS_ELAPSED_TIME` del **entrenamiento**, se rellenan con cada método y se mide el error absoluto medio (MAE) contra el valor real.

In [ ]:
# Se usa solo el 75 % más antiguo (el futuro entrenamiento): la prueba no participa.
corte = int(len(limpia) * (1 - config.FRACCION_PRUEBA))
entrenamiento_previo = limpia.iloc[:corte]
imputacion = comparar_imputadores(entrenamiento_previo["CRS_ELAPSED_TIME"],
                                  entrenamiento_previo["OP_UNIQUE_CARRIER"])
imputacion.to_csv(config.REPORTS_DIR / "comparacion_imputacion.csv", index=False)
imputacion

**Decisión:** la imputación de la duración vive **dentro del Pipeline** con `GroupMedianImputer` (mediana por aerolínea, `src/features/transformers.py`), de modo que en cada pliegue de validación se recalcula solo con los datos de entrenamiento de ese pliegue. Si una aerolínea no apareció en el entrenamiento, se usa la mediana global. El resto de columnas numéricas usa la mediana global (`SimpleImputer`) y las categóricas la etiqueta `"DESCONOCIDO"`.

### 3.4 Fugas de información (*data leakage*)
Una **fuga** es una columna que solo se conoce **después** del resultado. Con ella el modelo parece excelente en el notebook, pero en la vida real no se puede usar (como `decision_renovacion` en el ejemplo de seguros, que daba AUC = 1,000).

| Momento | Columnas | Decisión |
|---|---|:---:|
| **Antes del vuelo** (itinerario) | `FL_DATE`, `OP_UNIQUE_CARRIER`, `ORIGIN`, `DEST`, `CRS_DEP_TIME`, `CRS_ARR_TIME`, `CRS_ELAPSED_TIME`, `DISTANCE` | ✅ se usan |
| Al salir de puerta | `DEP_TIME`, `DEP_DELAY`, `DEP_DEL15`, `TAXI_OUT` | ❌ fuga |
| En el aire | `WHEELS_OFF`, `WHEELS_ON`, `AIR_TIME`, `ACTUAL_ELAPSED_TIME` | ❌ fuga |
| Al llegar / después | `ARR_TIME`, `ARR_DELAY`, `TAXI_IN`, `CARRIER_DELAY`, `WEATHER_DELAY`, `NAS_DELAY`, `SECURITY_DELAY`, `LATE_AIRCRAFT_DELAY` | ❌ fuga |
| Identificadores | `TAIL_NUM`, `OP_CARRIER_FL_NUM`, `ORIGIN_AIRPORT_ID`, `DEST_AIRPORT_ID` | ❌ no generalizan / redundantes |

**Demostración:** la misma regresión logística, con y sin la columna `DEP_DEL15` (¿salió tarde de la puerta?).

In [ ]:
# Columnas numéricas del itinerario (sin fuga) para una comparación rápida y justa.
base = ["CRS_DEP_TIME", "CRS_ARR_TIME", "CRS_ELAPSED_TIME", "DISTANCE", "MONTH", "DAY_OF_MONTH"]
con_fuga = base + ["DEP_DEL15"]                    # + una sola columna que se conoce al despegar

tr, va = limpia.iloc[:corte], limpia.iloc[corte:]  # pasado para entrenar, futuro para validar
for nombre, cols in [("Sin fuga (solo itinerario)", base), ("CON fuga (+ DEP_DEL15)", con_fuga)]:
    demo = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=1000))          # modelo sencillo
    demo.fit(tr[cols], tr[config.TARGET])                            # entrena
    auc = roc_auc_score(va[config.TARGET], demo.predict_proba(va[cols])[:, 1])
    print(f"{nombre:<28} ROC-AUC = {auc:.3f}")

**Lectura:** con `DEP_DEL15` el AUC se dispara porque *“si salió tarde, llega tarde”* — pero eso solo se sabe cuando el avión ya dejó la puerta. Por eso **todas** las columnas de fuga se eliminan antes de modelar. El AUC honesto es más bajo, y es el único que sirve en producción.

---
## 4 · Columnas: cuáles entran, cuáles se crean y cuáles se descartan

`crear_columnas()` agrega variables que **solo usan información de la misma fila** (no aprenden nada de otras filas, por eso no causan fuga):

| Columna nueva | De dónde sale | Por qué ayuda |
|---|---|---|
| `CRS_DEP_HOUR`, `CRS_ARR_HOUR` | `CRS_DEP_TIME` / `CRS_ARR_TIME` (HHMM → hora; 2400 → 0) | efecto cascada a lo largo del día |
| `DEP_TIME_BLOCK` | hora de salida | madrugada / mañana / tarde / noche |
| `CRS_DEP_SIN`, `CRS_DEP_COS` | hora en un círculo de 24 h | las 23 h quedan junto a las 0 h |
| `MONTH_SIN`, `MONTH_COS` | mes en un círculo de 12 | diciembre queda junto a enero |
| `DAY_OF_WEEK`, `IS_WEEKEND` | `FL_DATE` | demanda de negocios vs ocio |
| `SEASON` | mes | tormentas de verano, nieve de invierno |
| `ORIGIN_IS_HUB`, `DEST_IS_HUB`, `IS_HUB_TO_HUB` | lista fija de 10 hubs | congestión del aeropuerto |
| `DISTANCE_CAT` | `DISTANCE` | corta / media / larga |

In [ ]:
con_columnas = crear_columnas(limpia)              # agrega las columnas nuevas
tabla = descartar_fugas(con_columnas)              # quita fugas, identificadores y columnas vacías

# Tabla ANTES / DESPUÉS: qué columnas entraron, cuáles se crearon y cuáles se descartaron
originales, finales = set(muestra.columns), set(tabla.columns)
modelo_cols = set(config.COLUMNAS_MODELO)
filas = []
for col in sorted(originales | finales):
    if col in modelo_cols and col in originales:
        estado = "Entra (original)"
    elif col in modelo_cols:
        estado = "Entra (creada)"
    elif col in config.COLUMNAS_FUGA:
        estado = "Descartada (fuga)"
    elif col in config.COLUMNAS_ADMIN:
        estado = "Descartada (identificador)"
    elif col == config.TARGET:
        estado = "Etiqueta"
    else:
        estado = "Auxiliar (no entra al modelo)"
    filas.append({"columna": col, "estado": estado})
tabla_columnas = pd.DataFrame(filas).sort_values(["estado", "columna"]).reset_index(drop=True)
tabla_columnas.to_csv(config.REPORTS_DIR / "tabla_columnas.csv", index=False)
print(tabla_columnas["estado"].value_counts().to_string(), "\n")
display(tabla_columnas)

Las columnas **auxiliares** (`FL_DATE`, `CRS_DEP_TIME`, `CANCELLED`...) se conservan en la tabla para ordenar, filtrar o auditar, pero **no** entran al modelo: el modelo solo ve `config.COLUMNAS_MODELO`.

### 4.1 ¿Las columnas tienen relación con el retraso? (exploración)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
# (a) por hora programada de salida -> efecto cascada
tabla.groupby("CRS_DEP_HOUR")[config.TARGET].mean().plot(ax=ax[0, 0], marker="o")
ax[0, 0].set(title="Tasa de retraso por hora de salida", xlabel="Hora programada", ylabel="% retraso")
# (b) por mes -> estacionalidad
tabla.groupby("MONTH")[config.TARGET].mean().plot.bar(ax=ax[0, 1], color="#55A868")
ax[0, 1].set(title="Tasa de retraso por mes", xlabel="Mes", ylabel="% retraso")
# (c) por aerolínea -> modelo operativo de cada compañía
tabla.groupby("OP_UNIQUE_CARRIER")[config.TARGET].mean().sort_values().plot.barh(ax=ax[1, 0], color="#C44E52")
ax[1, 0].set(title="Tasa de retraso por aerolínea", xlabel="% retraso", ylabel="")
# (d) hub vs no hub en el origen -> congestión
tabla.groupby("ORIGIN_IS_HUB")[config.TARGET].mean().rename({0: "No hub", 1: "Hub"}).plot.bar(ax=ax[1, 1], color="#8172B2")
ax[1, 1].set(title="Tasa de retraso según el origen", xlabel="", ylabel="% retraso")
ax[1, 1].tick_params(axis="x", rotation=0)
porcentaje = plt.FuncFormatter(lambda v, _: f"{v:.0%}")             # 0.25 -> "25%"
for a in [ax[0, 0], ax[0, 1], ax[1, 1]]:
    a.yaxis.set_major_formatter(porcentaje)                          # eje vertical en porcentaje
ax[1, 0].xaxis.set_major_formatter(porcentaje)                       # barras horizontales: eje x
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "01_hipotesis_retraso.png", dpi=120)
plt.show()

### 4.2 El Pipeline: todas las transformaciones que *aprenden* van adentro
Imputar, escalar y hacer *one-hot* **aprenden** de los datos (medianas, medias, categorías). Si se hicieran antes de separar, la prueba contaminaría el entrenamiento. Dentro del `Pipeline`, `fit()` aprende solo con entrenamiento y `predict()` reutiliza lo aprendido.

- **Numéricas:** mediana → `StandardScaler` (misma escala; la logística lo necesita).
- **Categóricas:** `"DESCONOCIDO"` → `OneHotEncoder` que conserva las **25 categorías más frecuentes** y agrupa el resto (más de 300 aeropuertos) como *infrequent*; un aeropuerto nunca visto también cae ahí.

In [ ]:
# Así se ve el Pipeline completo con la regresión logística (diagrama interactivo):
crear_pipeline(LogisticRegression())

---
## 5 · Modelos: varios, comparados con la misma vara

### 5.1 Partición temporal (entrenamiento = pasado, prueba = futuro)
Los vuelos están ordenados por fecha. El **último 25 %** se guarda como **prueba** y **no se usa para elegir nada** (ni modelo, ni umbral, ni hiperparámetros). Un corte aleatorio entrenaría con 2026 para predecir 2025, algo imposible en la realidad.

In [ ]:
X_train, X_test, y_train, y_test = separar_temporal(tabla)   # 75 % más antiguo / 25 % más reciente
print("Columnas que ve el modelo:", X_train.shape[1])

### 5.2 Validación cruzada temporal y umbral de decisión
- **`TimeSeriesSplit` de 3 pliegues** dentro del entrenamiento: siempre se entrena con el pasado y se valida con lo que sigue.
- **Métrica para comparar: ROC-AUC** (qué tan bien ordena el riesgo; no depende del umbral y no la engaña el desbalance).
- **Umbral:** con `class_weight="balanced"` el 0,50 de `predict()` no es el mejor punto. Se elige, **en el último pliegue**, el umbral que maximiza el **F1 del retraso**.

### 5.3 Los modelos (sesiones 3, 4, 5 y 7)
| Modelo | Sesión | Idea |
|---|---|---|
| Línea base | — | siempre “puntual” |
| Regresión logística | 3 | frontera lineal sobre log-odds; coeficientes interpretables |
| Árbol de decisión | 4 | reglas si/entonces; captura interacciones |
| Random Forest | 5 | muchos árboles con *bagging*; reduce varianza |
| Gradient boosting | 5 | árboles en secuencia que corrigen residuos |
| XGBoost | 7 | boosting con regularización (λ, γ) |

**¿Por qué no KNN, SVM con kernel ni red neuronal?** Con ~57 mil filas y aeropuertos en one-hot, KNN y SVM-kernel recorren todo el entrenamiento en cada predicción (muy lentos para correr en vivo) y su frontera no aporta algo que no cubran la logística (lineal) y los ensambles (no lineal). Una red neuronal sobre datos tabulares de este tamaño no suele superar al boosting y es más difícil de explicar.

In [ ]:
modelos = definir_modelos(y_train)            # diccionario {nombre: modelo sin entrenar}

# Regla manual en el entrenamiento, como referencia (no tiene probabilidades -> no hay AUC)
print("Regla manual en entrenamiento:", metricas(y_train, regla_manual(X_train)))

# MISMA validación para todos: mismos datos, mismos pliegues, mismo Pipeline
filas = []
for nombre, modelo in modelos.items():
    resultado = validar_modelo(modelo, X_train, y_train)   # TimeSeriesSplit(3) + umbral
    filas.append({"modelo": nombre, **resultado})
    print(f"{nombre:<20} ROC-AUC = {resultado['roc_auc_cv']:.4f} ± {resultado['roc_auc_cv_std']:.4f}"
          f" | umbral = {resultado['umbral']:.2f} | {resultado['segundos_cv']} s")

tabla_cv = pd.DataFrame(filas).sort_values("roc_auc_cv", ascending=False).reset_index(drop=True)
tabla_cv

In [ ]:
graficar_comparacion_cv(tabla_cv, ruta=config.FIGURES_DIR / "02_comparacion_modelos_cv.png")
plt.show()

### 5.4 Elección del modelo (antes de mirar la prueba)
**Regla de decisión:** gana el mayor ROC-AUC de validación, **pero** si un modelo más simple queda a menos de **0,01**, se prefiere el simple (como Ridge vs XGBoost en el ejemplo de las casas): es más rápido, más estable y se puede explicar con sus coeficientes.

In [ ]:
campeon, argumento = elegir_campeon(tabla_cv)
print("MODELO ELEGIDO:", campeon)
print(argumento)

### 5.5 Hiperparámetros: búsqueda pequeña y honesta
Se ajustan el **campeón** y el de **mayor AUC** con `GridSearchCV` (4 combinaciones, `TimeSeriesSplit` de 2 pliegues, solo entrenamiento). Para cambiar la configuración se exige una ganancia de **al menos 0,005** de AUC: ganancias menores están dentro del ruido entre pliegues.

In [ ]:
candidatos = [campeon]
mejor_auc = tabla_cv[tabla_cv["modelo"] != "Línea base"].iloc[0]["modelo"]
if mejor_auc != campeon:
    candidatos.append(mejor_auc)                  # también se ajusta el de mayor AUC

ajustes = [ajustar_hiperparametros(n, modelos[n], X_train, y_train) for n in candidatos]
tabla_ajuste = pd.DataFrame([{k: v for k, v in a.items() if k != "estimador"} for a in ajustes])
tabla_ajuste.to_csv(config.MODELS_DIR / "hiperparametros.csv", index=False)
display(tabla_ajuste)

# Si algún ajuste supera la ganancia mínima, se adopta y se recalcula su umbral con la misma CV.
for a in ajustes:
    if a["se_adopta"]:
        modelos[a["modelo"]] = a["estimador"]
        nuevo = validar_modelo(a["estimador"], X_train, y_train)
        for k, v in nuevo.items():
            tabla_cv.loc[tabla_cv["modelo"] == a["modelo"], k] = v
if any(a["se_adopta"] for a in ajustes):         # con AUC nuevos, la elección se rehace
    tabla_cv = tabla_cv.sort_values("roc_auc_cv", ascending=False).reset_index(drop=True)
    campeon, argumento = elegir_campeon(tabla_cv)
print("Modelo elegido tras el ajuste:", campeon)

---
## 6 · Evaluación final en la prueba (datos que ningún modelo ha visto)

Cada modelo se entrena con **todo** el entrenamiento y se evalúa en la prueba con **su umbral ya fijado** en la validación. La tabla incluye la **línea base** y la **regla manual**.

> ⚠️ La **exactitud** engaña: decir siempre “puntual” acierta ~78 % con recall 0. Por eso se reportan F1, precisión, recall y AUC del retraso.

In [ ]:
filas, pipelines, probabilidades, predicciones = [], {}, {}, {}
for nombre, modelo in modelos.items():
    pipe = crear_pipeline(clone(modelo)).fit(X_train, y_train)       # entrena con todo el entrenamiento
    umbral = float(tabla_cv.loc[tabla_cv["modelo"] == nombre, "umbral"].iloc[0])  # umbral de la CV
    prob = pipe.predict_proba(X_test)[:, 1]                          # probabilidad de retraso
    pred = (prob >= umbral).astype(int)                              # decisión con ese umbral
    fila_cv = tabla_cv[tabla_cv["modelo"] == nombre].iloc[0]
    filas.append({"modelo": nombre, "roc_auc_cv": fila_cv["roc_auc_cv"],
                  "roc_auc_cv_std": fila_cv["roc_auc_cv_std"], "umbral": umbral,
                  **metricas(y_test, pred, prob)})
    pipelines[nombre], probabilidades[nombre], predicciones[nombre] = pipe, prob, pred

pred_regla = regla_manual(X_test)                                    # la regla a mano en la prueba
filas.append({"modelo": "Regla manual", "roc_auc_cv": np.nan, "roc_auc_cv_std": np.nan,
              "umbral": np.nan, **metricas(y_test, pred_regla)})

resultados = pd.DataFrame(filas)
resultados.to_csv(config.MODELS_DIR / "metrics_comparison.csv", index=False)

# Se marca el modelo elegido para que se vea en la exposición
(resultados.style.format(precision=4, na_rep="—")
           .apply(lambda r: ["background-color: #d4edda" if r["modelo"] == campeon else "" for _ in r], axis=1))

In [ ]:
# Curvas ROC y Precisión-Recall en la prueba (sin la línea base, que es una recta)
curvas = {k: v for k, v in probabilidades.items() if k != "Línea base"}
graficar_curvas(y_test, curvas, ruta=config.FIGURES_DIR / "03_curvas_prueba.png")
plt.show()

# Matrices de confusión: modelo elegido vs regla manual
graficar_matrices(y_test, {f"{campeon} (umbral {resultados.set_index('modelo').loc[campeon, 'umbral']:.2f})":
                           predicciones[campeon], "Regla manual": pred_regla},
                  ruta=config.FIGURES_DIR / "04_matrices_confusion.png")
plt.show()

### 6.1 ¿Qué columnas pesan más en el modelo elegido?

In [ ]:
importancias = importancia_columnas(pipelines[campeon], n=15)
importancias.to_csv(config.REPORTS_DIR / "importancia_columnas.csv", index=False)
display(importancias)

En la regresión logística, un **coeficiente positivo** sube la probabilidad de retraso y `odds_ratio = e^coef` dice por cuánto se multiplican los *odds* cuando la variable (escalada) sube una desviación estándar, o cuando la categoría está presente. Recordatorio de la sesión 3: es **asociación, no causa**.

### 6.2 Conclusión escrita con los números de esta corrida

In [ ]:
r = resultados.set_index("modelo")
c, regla, base = r.loc[campeon], r.loc["Regla manual"], r.loc["Línea base"]
texto = f'''MODELO ELEGIDO: {campeon} (umbral {c['umbral']:.2f})

{argumento}

En la PRUEBA (no usada para elegir):
  - El modelo detecta el {c['recall_retraso']:.0%} de los retrasos (recall) con precisión {c['precision_retraso']:.2f}:
    de cada 100 alertas, {c['precision_retraso']*100:.0f} son retrasos reales.
  - F1 del retraso: {c['f1_retraso']:.4f}  vs  regla manual {regla['f1_retraso']:.4f}  vs  línea base {base['f1_retraso']:.4f}.
  - ROC-AUC en prueba: {c['roc_auc']:.4f} (validación: {c['roc_auc_cv']:.4f} ± {c['roc_auc_cv_std']:.4f}).
  - La línea base acierta {base['exactitud']:.1%} sin detectar ni un retraso: la exactitud no sirve para decidir.'''
print(texto)
_ = (config.MODELS_DIR / "eleccion.txt").write_text(texto, encoding="utf-8")  # "_ =" evita imprimir el número de caracteres

### 6.3 Guardar el modelo y usarlo con un vuelo nuevo

In [ ]:
import joblib                                                     # serializa objetos de Python
paquete = {"pipeline": pipelines[campeon],                        # preprocesamiento + modelo
           "umbral": float(r.loc[campeon, "umbral"]),             # punto de operación
           "modelo": campeon}
joblib.dump(paquete, config.MODELS_DIR / "best_model.joblib")
print("Guardado en", config.MODELS_DIR / "best_model.joblib")

# Ejemplo: un vuelo programado de Atlanta a Chicago O'Hare, viernes 18:30 de julio.
vuelo = pd.DataFrame([{"FL_DATE": "7/17/2026 12:00:00 AM", "OP_UNIQUE_CARRIER": "DL",
                       "ORIGIN": "ATL", "DEST": "ORD", "CRS_DEP_TIME": 1830, "CRS_ARR_TIME": 1945,
                       "CRS_ELAPSED_TIME": 135, "DISTANCE": 606}])
vuelo["FL_DATE"] = convertir_fecha(vuelo["FL_DATE"])            # texto -> fecha, igual que en la limpieza
vuelo = crear_columnas(vuelo)[config.COLUMNAS_MODELO]             # mismas columnas que en el entrenamiento
p = paquete["pipeline"].predict_proba(vuelo)[0, 1]
print(f"Probabilidad de retraso: {p:.1%} -> {'ALERTA de retraso' if p >= paquete['umbral'] else 'sin alerta'}"
      f" (umbral {paquete['umbral']:.2f})")

---
## 7 · Limitaciones y siguientes pasos
- **Solo información del itinerario.** Sin clima pronosticado ni el estado del avión que llega (*retraso del vuelo anterior*), el techo de AUC es moderado (~0,6–0,65). Unir el pronóstico del tiempo por aeropuerto (NOAA) o la rotación de la aeronave (`TAIL_NUM` del vuelo previo, conocido horas antes) sería el siguiente salto.
- **Muestra del 1 %.** Suficiente para comparar modelos de forma justa y correr en vivo; con más datos los ensambles podrían separarse más de la logística.
- **Umbral orientado a recall.** Maximizar F1 deja un recall alto con precisión baja (muchas falsas alarmas). Si el costo de una falsa alarma fuera alto, el umbral se movería hacia arriba.